Install and Import Required Libraries
Run this in your first cell to install any necessary scraping and machine learning libraries.

In [10]:
# Run this cell to install required libraries if not already installed
%pip install pandas scikit-learn requests beautifulsoup4 selenium webdriver-manager

     --------------------------------------- 11.8/11.8 MB 62.1 kB/s eta 0:00:00
     --------------------------------------- 95.8/95.8 KB 88.3 kB/s eta 0:00:00
     ------------------------------------- 511.8/511.8 KB 81.9 kB/s eta 0:00:00
     ------------------------------------ 185.1/185.1 KB 119.1 kB/s eta 0:00:00
     --------------------------------------- 51.2/51.2 KB 90.4 kB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
You should consider upgrading via the 'c:\Users\DELL\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.



Download Directly in Your Notebook

Once the environment variables are injected, the official Python library automatically authenticates without requiring a kaggle.json file. Run this script directly inside your OpenShift AI notebook cell:

In [2]:
import os
import subprocess

# 1. Install the Kaggle CLI tool
subprocess.run(["pip", "install", "--quiet", "kaggle"])

# 2. Define the dataset slug (e.g., 50k IMDb reviews)
dataset_slug = "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"
download_path = "./data"

# 3. Execute the secure download and unzip
os.makedirs(download_path, exist_ok=True)
subprocess.run([
    "kaggle", "datasets", "download", 
    "-d", dataset_slug, 
    "-p", download_path, 
    "--unzip"
])

print(f"Dataset downloaded successfully to: {download_path}")


Dataset downloaded successfully to: ./data


Load Kaggle Dataset

In [11]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Load the 50K IMDB dataset from Kaggle
# Make sure 'IMDB Dataset.csv' is in the same directory as your notebook
df_kaggle = pd.read_csv('./data/IMDB Dataset.csv')

# Convert text labels ('positive'/'negative') to binary (1/0)
df_kaggle['label'] = df_kaggle['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)

print("Dataset shape:", df_kaggle.shape)
df_kaggle.head()

Dataset shape: (50000, 3)


,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


Train the Sentiment Analysis Model
We train the model using the 50K dataset with TF-IDF vectorization and Logistic Regression.

In [12]:
X = df_kaggle['review']
y = df_kaggle['label']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vectorize text using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_features=10000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train the model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

# Evaluate model performance
preds = model.predict(X_test_vec)
print(f"Model Accuracy: {accuracy_score(y_test, preds) * 100:.2f}%")

Model Accuracy: 89.10%


Define IMDb URI Review Scraper
Since IMDb doesn't offer a direct free public API for reviews, this function scrapes the review page of any given IMDb movie URI.

In [13]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def scrape_imdb_movie_reviews(imdb_uri):
    # Ensure the URI points to the reviews sub-page
    if not imdb_uri.endswith("/reviews"):
        imdb_uri = imdb_uri.rstrip("/") + "/reviews"
        
    # Configure headless Chrome browser
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")  # Run without opening a visible browser window
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    # Initialize driver
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    reviews = []
    try:
        print(f"Loading page and waiting for content: {imdb_uri}")
        driver.get(imdb_uri)
        
        # Explicit wait: Wait up to 10 seconds until review containers are present on the page
        wait = WebDriverWait(driver, 10)
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, 'review-container')))
        
        # Extract review containers
        containers = driver.find_elements(By.CLASS_NAME, 'review-container')
        
        for container in containers:
            try:
                # Find the text element inside each container
                review_elem = container.find_element(By.CLASS_NAME, 'text')
                if review_elem:
                    reviews.text = review_elem.text.strip()
                    if review_text:
                        reviews.append(review_text)
            except Exception:
                continue
                
    except Exception as e:
        print(f"An error occurred while waiting for content: {e}")
    finally:
        # Always close the browser instance
        driver.quit()
        
    return reviews

Predict Movie Category (Good or Bad) Using an IMDb URI
This function combines the scraper and your trained model to analyze an IMDb movie link and output an overall verdict.

In [20]:
def classify_movie_from_uri(imdb_uri, model, vectorizer):
    print(f"Fetching and analyzing reviews for: {imdb_uri}")
    reviews = ['''It is no wonder that the film has such a high rating, it is quite literally breathtaking. What can I say that hasn't said before? Not much, it's the story, the acting, the premise, but most of all, this movie is about how it makes you feel. Sometimes you watch a film, and can't remember it days later, this film loves with you, once you've seen it, you don't forget.

The ultimate story of friendship, of hope, and of life, and overcoming adversity.

I understand why so many class this as the best film of all time, it isn't mine, but I get it. If you haven't seen it, or haven't seen it for some time, you need to watch it, it's amazing. 10/10.''',
'''Don't get me wrong, Jenna Ortega's performance was absolutely fantastic. She captures Wednesday's essence perfectly. Pretty much everything else about the show falls flat for me. The plot really doesn't work for me. It reminds me of the Chilling Adventures of Sabrina and Riverdale. It doesn't feel creative or inventive at all and just doesn't work for me. The whole "school for outsiders" thing plus murder mystery just wasn't a hit in my opinion. I also wasn't impressed with the casting of Morticia and Gomez and their relationship with Wednesday. Overall, it feels very YA so maybe it's just meant for a younger crowd.''',
'''It was watchable but I didn't love it.

I'm not sure just how involved Burton actually was with this project, it doesn't feel like he was massively involved because this series really didn't scream "Burton-esque" to me. He supposedly directed the series, but a few different directors' names pop up in the credits?

I was excited to see it when I found out Tim's name was attached to it, but it ended up being far too "teeny high school drama" for my liking.

I did like Jenna Ortega as Wednesday, I thought she portrayed the character well and made it her own. Uncle Fester was okay as well. And Thing, but, ya know, he's a hand. The casting for the rest of the Addams however, I didn't like; they just didn't capture the essence of the characters in my eyes.

This series had its moments, but as a whole I just didn't get what I was hoping for and was left a bit disappointed.''']
    #reviews = scrape_imdb_movie_reviews(imdb_uri)
    
    if not reviews:
        return "No reviews found or access was blocked by IMDb."
    
    # Vectorize and predict sentiment for each review
    vec_reviews = vectorizer.transform(reviews)
    predictions = model.predict(vec_reviews)
    
    # Calculate percentage of positive (good) reviews
    positive_count = sum(predictions)
    total_reviews = len(predictions)
    positive_ratio = (positive_count / total_reviews) * 100
    
    # Determine overall movie category
    verdict = "Good Movie 😊" if positive_ratio >= 50 else "Bad Movie 😞"
    
    print(f"\n--- Results ---")
    print(f"Total Reviews Analyzed: {total_reviews}")
    print(f"Positive Reviews: {positive_ratio:.1f}%")
    print(f"Overall Categorization: {verdict}\n")
    
    # Return sample breakdown DataFrame
    results_df = pd.DataFrame({
        "Review": reviews,
        "Prediction": ["Good" if p == 1 else "Bad" for p in predictions]
    })
    return results_df

# --- Example Usage ---
movie_uri = "https://www.imdb.com/title/tt0111161/" # The Shawshank Redemption
result_df = classify_movie_from_uri(movie_uri, model, vectorizer)
print("result_df:")
print(result_df)

Fetching and analyzing reviews for: https://www.imdb.com/title/tt0111161/

--- Results ---
Total Reviews Analyzed: 3
Positive Reviews: 66.7%
Overall Categorization: Good Movie 😊

result_df:
                                              Review Prediction
0  It is no wonder that the film has such a high ...       Good
1  Don't get me wrong, Jenna Ortega's performance...       Good
2  It was watchable but I didn't love it.\n\nI'm ...        Bad
